# Kayıp Fonksiyonları

Bu alıştırmada, Kayıp fonksiyonlarının `LinearRegression` modeli üzerindeki etkilerini karşılaştıracaksınız.

👇 Bu zorluk için kullanmak üzere bir CSV dosyası indirelim ve onu bir DataFrame'e dönüştürelim

In [1]:
import pandas as pd

data = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/loss_functions_dataset.csv")
data.sample(5)

,Relative Compactness,Surface Area,Wall Area,Roof Area,Overall Height,Glazing Area,Average Temperature
280,0.64,784.0,343.0,220.5,3.5,0.10,17.200
267,0.74,686.0,245.0,220.5,3.5,0.10,11.945
519,0.66,759.5,318.5,220.5,3.5,0.25,14.540
441,0.86,588.0,294.0,147.0,7.0,0.25,31.305
82,0.69,735.0,294.0,220.5,3.5,0.10,12.695


🎯 Göreviniz, tasarımına göre bir seranın içindeki ortalama sıcaklığı tahmin etmektir. Sıcaklık tahminleriniz, her bir bitki için iklim ihtiyaçlarına göre uygun sera tasarımını seçmenize yardımcı olacaktır.

🌿 Bitkilerin küçük sıcaklık değişimlerini kaldırabildiğini, ancak sıcaklık değişimleri arttıkça katlanarak daha duyarlı hale geldiğini biliyorsunuz.

## 1. Teori

❓ Teorik olarak, bitkileri öldürme riskini sınırlamak için modelinizi hangi Kayıp fonksiyonu üzerinde eğitirsiniz?

<details>
<summary> 🆘 Cevap </summary>
    
Teorik olarak, Ortalama Kare Hata (MSE) Kayıp fonksiyonunu kullanırsınız. Bu, aykırı tahminleri cezalandırır ve modelinizin büyük hatalar yapmasını engeller. Bu, daha küçük sıcaklık değişimleri ve bitkiler için daha düşük risk sağlayacaktır.

</details>

**MSE (Mean Squared Error).**

Bitkiler küçük sapmalara dayanıklı, büyük sapmalarda katlanarak duyarlı. Kayıp fonksiyonunun
ceza eğrisi bu zarar eğrisine benzemeli. MSE hatayı kareyle cezalandırdığı için 6 °C'lik tek
bir sapma, 2 °C'lik üç sapmadan çok daha pahalıya gelir — model de ağırlıklarını bu büyük
sapmayı kaçınmak üzere ayarlar.

MAE tüm hataları lineer cezalandırır; ortalamayı iyi tutar ama tek bir aşırı sapmayı önleme
konusunda MSE kadar baskı uygulamaz. Burada riskin kaynağı ortalama hata değil, en kötü
durumdaki hata.

## 2. Uygulama

### 2.1 Ön İşleme

❓ Özellikleri standartlaştırın

In [2]:
from sklearn.preprocessing import StandardScaler

X = data.drop(columns="Average Temperature")
y = data["Average Temperature"]

X_scaled = pd.DataFrame(StandardScaler().fit_transform(X), columns=X.columns)
X_scaled.describe().round(2)

,Relative Compactness,Surface Area,Wall Area,Roof Area,Overall Height,Glazing Area
count,768.00,768.00,768.00,768.00,768.0,768.00
mean,-0.00,-0.00,0.00,0.00,0.0,0.00
std,1.00,1.00,1.00,1.00,1.0,1.00
min,-1.36,-1.79,-1.69,-1.47,-1.0,-1.76
25%,-0.77,-0.74,-0.56,-0.79,-1.0,-1.01
50%,-0.13,0.02,0.00,0.16,0.0,0.12
75%,0.62,0.79,0.56,0.97,1.0,1.24
max,2.04,1.55,2.25,0.97,1.0,1.24


### 2.2 Modelleme

Bu bölümde, farklı Kayıp fonksiyonları üzerinde optimize edilmiş modelleri değerlendirerek teoriyi doğrulayacaksınız.

### En Küçük Kareler (MSE) Kaybı

❓ **En Küçük Kareler Kaybı** (MSE) üzerinde **Stokastik Gradyan İnişi** (SGD) ile optimize edilmiş bir Doğrusal Regresyon modelini **10-Katlı Çapraz doğrula**

In [3]:
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import cross_validate

cv_mse = cross_validate(
    SGDRegressor(loss="squared_error", random_state=42),
    X_scaled, y,
    cv=10,
    scoring=["r2", "max_error"]
)

❓ Hesaplayın:
- Ortalama çapraz doğrulanmış R2 skoru ve bunu `r2` değişkeninde kaydedin
- Tüm katlarınızın °C cinsinden en büyük tek tahmin hatasını hesaplayın ve `max_error_celsius` değişkeninde kaydedin

(İpucu: `max_error` sklearn'de kabul edilen bir puanlama metriğidir)

In [4]:
r2 = cv_mse["test_r2"].mean()
max_error_celsius = abs(cv_mse["test_max_error"]).max()

print(f"MSE loss -> r2: {r2:.3f}   max error: {max_error_celsius:.2f} °C")

MSE loss -> r2: 0.898   max error: 9.79 °C


### Ortalama Mutlak Hata (MAE) Kaybı

Peki modelimizi MAE üzerinde optimize edersek ne olur?

❓ **MAE** Kaybı üzerinde **Stokastik Gradyan İnişi** (SGD) ile optimize edilmiş bir Doğrusal Regresyon modelini **10-Katlı Çapraz doğrula**

<details>
<summary>💡 İpuçları</summary>

- MAE kaybı `SGDRegressor`'da doğrudan belirtilemez. Doğru parametreleri ayarlayarak tasarlanması gerekir

</details>

In [5]:
cv_mae = cross_validate(
    SGDRegressor(loss="epsilon_insensitive", epsilon=0, random_state=42),
    X_scaled, y,
    cv=10,
    scoring=["r2", "max_error"]
)

❓ Hesaplayın:
- Ortalama çapraz doğrulanmış R2 skoru, bunu `r2_mae`'de saklayın
- Tüm katlarınızın en büyük tek tahmin hatasını, bunu `max_error_mae`'de saklayın

In [ ]:
# YOUR CODE HERE

## 3. Sonuç

❓ Değerlendirdiğiniz modellerden hangisi göreviniz için en uygun görünüyor?

<details>
<summary> 🆘Cevap </summary>
    
İki model arasında ortalama çapraz doğrulanmış r2 skorları yaklaşık olarak benzer olmasına rağmen, MAE üzerinde optimize edilen modelin zaman zaman daha büyük hatalar yapma şansı daha fazladır, bu da bitkileri öldürme riskini artırır!
    
</details>

> CEVABINIZI BURAYA YAZIN

# 🏁 Kodunuzu kontrol edin ve notebook'unuzu gönderin

In [ ]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'loss_functions',
    r2 = r2,
    r2_mae = r2_mae,
    max_error = max_error_celsius,
    max_error_mae = max_error_mae
)

result.write()
print(result.check())